## DATA PREPROCESSING

## Inference for the Import Code
First, I imported Pandas and NumPy for data manipulation and numerical operations. Then I imported Scikit-learn functions for splitting the dataset, encoding categorical variables, scaling numerical features, and building a preprocessing pipeline. These tools prepare the dataset so that it can be used effectively for machine learning

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
file_path = "ev_vs_petrol_dataset_v3.csv"

df = pd.read_csv(file_path)

## Check Missing Values
I checked for missing values in every column. The result showed zero missing values across the dataset, so no missing-value treatment was required before continuing with preprocessing.

In [3]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
country                          0
region                           0
year                             0
vehicle_segment                  0
powertrain_type                  0
ev_sales                         0
petrol_car_sales                 0
diesel_car_sales                 0
total_vehicle_sales              0
ev_market_share                  0
charging_stations                0
fast_chargers_share              0
avg_ev_range_km                  0
fuel_price_usd_per_liter         0
electricity_price_usd_per_kwh    0
gdp_per_capita                   0
urban_population_percent         0
co2_emissions_transport_mt       0
ev_subsidy_usd                   0
emission_regulation_score        0
ev_growth_rate_yoy               0
is_ev_dominant                   0
dtype: int64


## Missing Count
I calculated both the number and percentage of missing values for every column. The result showed 0 missing values and 0% missing data across all columns. Therefore, the dataset does not require any missing-value treatment.

In [4]:
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df)) * 100
})

display(missing)

,Missing Count,Missing %
country,0,0.0
region,0,0.0
year,0,0.0
vehicle_segment,0,0.0
powertrain_type,0,0.0
ev_sales,0,0.0
petrol_car_sales,0,0.0
diesel_car_sales,0,0.0
total_vehicle_sales,0,0.0
ev_market_share,0,0.0


## Remove Duplicates
I checked the dataset for duplicate records using duplicated(). The result was zero, confirming that there are no duplicate rows and the dataset does not require duplicate removal.

In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


No duplicate rows were present, so the dataset remained unchanged at 1,200 × 22 after duplicate removal.

In [6]:
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (1200, 22)


## Handle Missing Values
I separated the dataset into numerical and categorical columns based on their data types. This is necessary because numerical variables and categorical variables require different preprocessing techniques. Numerical features will be scaled, while categorical features will be converted into numerical values using one-hot encoding.

In [7]:
numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = df.select_dtypes(
    include="object"
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['year', 'ev_sales', 'petrol_car_sales', 'diesel_car_sales', 'total_vehicle_sales', 'ev_market_share', 'charging_stations', 'fast_chargers_share', 'avg_ev_range_km', 'fuel_price_usd_per_liter', 'electricity_price_usd_per_kwh', 'gdp_per_capita', 'urban_population_percent', 'co2_emissions_transport_mt', 'ev_subsidy_usd', 'emission_regulation_score', 'ev_growth_rate_yoy', 'is_ev_dominant']

Categorical columns:
['country', 'region', 'vehicle_segment', 'powertrain_type']


C:\Users\VICKY\AppData\Local\Temp\ipykernel_4596\964793171.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


## Fill numerical missing values with median

Median imputation was applied as a precaution for numerical features; because there are no missing values, the dataset remains unchanged.

In [8]:
for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

## Fill categorical missing values with mode

This step applies mode imputation to categorical variables as a precautionary preprocessing step. Since your missing-value analysis showed 0 missing values across all columns, no categorical values are actually replaced in the current dataset. Therefore, the dataset remains unchanged after this operation

In [9]:
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

The final validation confirms that there are 0 missing values remaining in the dataset.

In [10]:
print("Total missing values:",
      df.isnull().sum().sum())

Total missing values: 0


## target and feauture separation

ev_sales is defined as the target, while target-related variables are removed from the input features to prevent data leakage before model training.

In [11]:
target = "ev_sales"

X = df.drop(
    columns=[
        "ev_sales",
        "ev_market_share",
        "is_ev_dominant"
    ]
)

y = df["ev_sales"]

The feature and target datasets are correctly structured and aligned, with 1,200 observations available for predicting EV sales from 19 input features.

In [12]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

X shape: (1200, 19)
y shape: (1200,)

Features:
['country', 'region', 'year', 'vehicle_segment', 'powertrain_type', 'petrol_car_sales', 'diesel_car_sales', 'total_vehicle_sales', 'charging_stations', 'fast_chargers_share', 'avg_ev_range_km', 'fuel_price_usd_per_liter', 'electricity_price_usd_per_kwh', 'gdp_per_capita', 'urban_population_percent', 'co2_emissions_transport_mt', 'ev_subsidy_usd', 'emission_regulation_score', 'ev_growth_rate_yoy']

Target:
ev_sales


## Train-Test Split

I split the dataset into 80% training data and 20% testing data. The training set contains 960 records and the testing set contains 240 records. The model will learn patterns from the training data and its performance will then be evaluated on the unseen testing data. I used random state 42 to ensure reproducibility.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training data: (960, 19)
Testing data: (240, 19)
Training target: (960,)
Testing target: (240,)


## Identify Numerical and Categorical Features
After splitting the dataset, I identified the numerical and categorical features using only the training data. Numerical features will be scaled using StandardScaler, while categorical features will be converted into numerical form using OneHotEncoder. This prepares the different types of features for machine-learning models

In [14]:
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include="object"
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['year', 'petrol_car_sales', 'diesel_car_sales', 'total_vehicle_sales', 'charging_stations', 'fast_chargers_share', 'avg_ev_range_km', 'fuel_price_usd_per_liter', 'electricity_price_usd_per_kwh', 'gdp_per_capita', 'urban_population_percent', 'co2_emissions_transport_mt', 'ev_subsidy_usd', 'emission_regulation_score', 'ev_growth_rate_yoy']

Categorical features:
['country', 'region', 'vehicle_segment', 'powertrain_type']


C:\Users\VICKY\AppData\Local\Temp\ipykernel_4596\562158035.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


## Encoding
I used OneHotEncoder to convert categorical variables into numerical binary features. I set handle_unknown to ignore so that new categories in the test data do not cause errors. I also used sparse_output=False to obtain a regular numerical array.

In [15]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

## Scaling
I initialized StandardScaler to standardize the numerical features. Since my dataset contains variables with very different ranges, scaling brings them to a comparable scale and helps prevent large-valued features from dominating the model.

In [16]:
scaler = StandardScaler()

## Preprocessing Pipeline
I created a ColumnTransformer to apply the appropriate preprocessing technique to each feature type. Numerical features are standardized using StandardScaler, while categorical features are converted into numerical form using OneHotEncoder. Combining these steps in one transformer makes the preprocessing process consistent and suitable for machine learning.

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

## fit_transform() on Training Data
I used fit_transform() on the training data. The preprocessor first learns the scaling parameters and categorical categories from the training data and then transforms the data. This ensures that preprocessing is learned only from the training set and helps prevent data leakage.

In [18]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

## Transform test data
I transformed the test data using the preprocessing rules learned from the training data. I used transform() instead of fit_transform() so that the test data does not influence the scaling or encoding process. This prevents data leakage and ensures a fair evaluation of the machine-learning model.

In [19]:
X_test_processed = preprocessor.transform(
    X_test
)

## Processed Training and Testing Shapes
After preprocessing, I checked the shapes of both datasets. The training data contains 960 records and the testing data contains 240 records, and both have 50 processed features. The increase from 19 to 50 features is due to one-hot encoding of categorical variables.

In [20]:
print("Processed training shape:",
      X_train_processed.shape)

print("Processed testing shape:",
      X_test_processed.shape)

Processed training shape: (960, 50)
Processed testing shape: (240, 50)


## Creating Feature Names and DataFrame
After preprocessing, I converted the transformed arrays back into Pandas DataFrames and assigned meaningful feature names using get_feature_names_out(). This makes the processed data easier to inspect and ensures that the training and testing datasets have the same feature structure

In [21]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

The first five rows verify that the training dataset has been successfully scaled, encoded, labelled, and converted into a machine-learning-ready format.

In [22]:
X_train_processed_df.head()

,num__year,num__petrol_car_sales,num__diesel_car_sales,num__total_vehicle_sales,num__charging_stations,num__fast_chargers_share,num__avg_ev_range_km,num__fuel_price_usd_per_liter,num__electricity_price_usd_per_kwh,num__gdp_per_capita,...,cat__region_APAC,cat__region_Europe,cat__region_North America,cat__region_Oceania,cat__region_South America,cat__vehicle_segment_commercial,cat__vehicle_segment_mass_market,cat__vehicle_segment_premium,cat__powertrain_type_BEV,cat__powertrain_type_ICE
331,1.435849,-0.013690,-0.140208,0.072247,0.286037,0.543847,1.658553,1.558100,0.908645,0.370139,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
409,0.125776,0.731697,0.454654,0.627987,-0.162839,-0.403926,-0.137591,-0.738649,-1.118229,-1.396963,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
76,0.344122,-0.345337,-0.404883,-0.358544,-0.140850,0.244550,0.367316,-0.052643,0.591946,0.572213,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
868,-1.402642,-0.162862,-0.259256,-0.193888,-0.164693,-1.162144,-1.114296,-0.398390,0.287915,-0.231371,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
138,1.435849,-0.420530,-0.397117,-0.418249,-0.057410,0.663566,1.559227,1.640421,2.023425,0.702835,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


## Final preprocessing check

Finally, I performed a complete preprocessing validation. The dataset contains 1,200 records, which were divided into 960 training and 240 testing records. After preprocessing, both datasets contain 50 features. There are no missing values or duplicate records, confirming that the data has been successfully prepared and is ready for machine-learning model development.

In [23]:
print("\nOriginal dataset:", df.shape)

print("Training data:", X_train.shape)

print("Testing data:", X_test.shape)

print("Processed training data:",
      X_train_processed.shape)

print("Processed testing data:",
      X_test_processed.shape)

print("\nMissing values in original dataset:",
      df.isnull().sum().sum())

print("\nDuplicate rows:",
      df.duplicated().sum())


Original dataset: (1200, 22)
Training data: (960, 19)
Testing data: (240, 19)
Processed training data: (960, 50)
Processed testing data: (240, 50)

Missing values in original dataset: 0

Duplicate rows: 0


## Data Preprocessing Conclusion

The data preprocessing stage successfully prepared the EV and Petrol vehicle dataset for machine-learning model development. The dataset was checked for missing values and duplicate records, and the results confirmed that there were **no missing values and no duplicate rows**.

The target variable, **`ev_sales`**, was clearly separated from the input features. Target-related variables such as **`ev_market_share`** and **`is_ev_dominant`** were excluded to reduce the risk of target leakage. The dataset was then divided into **80% training data (960 records)** and **20% testing data (240 records)**.

Numerical and categorical features were identified separately. Numerical features were standardized using **StandardScaler**, while categorical features were converted into numerical representations using **OneHotEncoder**. The preprocessing transformation increased the feature count from **19 original input features to 50 processed features**.

The final preprocessing validation confirmed that the training and testing datasets were transformed consistently, with **960 × 50** processed training data and **240 × 50** processed testing data. Therefore, the dataset is clean, properly structured, and ready for the **machine-learning modeling stage**.
